In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

### **Data Reading**

In [0]:
df_customers = spark.sql("select * from databricks_ete_project.bronze.bronze_customers")
display(df_customers)

In [0]:
df_customers = df_customers.withColumn('domains', split(col('email'), '@')[1])

In [0]:
df_customers.display()

In [0]:
df_customers.groupBy('domains').agg(count('customer_id').alias('total_customers')).sort('total_customers', ascending=False).display()

In [0]:
df_gmail = df_customers.filter(col('domains') == 'gmail.com')
df_gmail.display()


In [0]:
df_yahoo = df_customers.filter(col('domains') == 'yahoo.com')
df_yahoo.display()

In [0]:
df_hotmail = df_customers.filter(col('domains') == 'hotmail.com')
df_hotmail.display()

In [0]:
df_customers = df_customers.withColumn('full_name', concat(col('first_name'), lit(' '), col('last_name')))\
                           .drop('first_name', "last_name")

df_customers.display()

In [0]:
df_customers.write.format("delta").mode("overwrite").option('overwriteSchema', 'true').saveAsTable("databricks_ete_project.silver.silver_customers")

In [0]:
%sql
select * from databricks_ete_project.silver.silver_customers